# Model overview: every model, what is SINDy and what is not, and predictions against real battles

All world models of the SINDYc line, on the same exp06 validation battles (1,000 close battles,
Blue's intent switching every 1-4 s, never used for fitting), plus a stalker battle for transfer.

**How to read the labels.** Every part of a model is one of:

| label | meaning |
|---|---|
| 🟢 **SINDy** | chosen by sparse regression (STLSQ): which terms survive *and* their coefficients |
| 🟡 **designed, fitted outside SINDy** | I wrote the formula; `curve_fit` fitted its numbers before SINDy ran; nothing removes unneeded parts |
| 🔵 **game rule** | a constant taken from SMAClite (reach, speed, health), not fitted |
| ⚪ **designed, no parameters** | a modelling choice with no fitted numbers (one-hot intents, holding the formation fixed) |

Sections: 1 the models · 2 anatomy of the current model · 3 whole battles · 4 rolling forecasts ·
5 errors with intervals · 6 where it goes wrong (units in reach) · 7 counterfactuals · 8 stalkers.

In [ ]:
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysindy as ps

from sindy_wm.envs.intents import INTENTS
from sindy_wm.models.intent_model import (
    CTRL,
    DT,
    REACH_COLS,
    SHAPE,
    STATE,
    IntentModel,
    add_controls,
    add_rates,
    control_vector,
    format_terms,
    horizon_errors,
    make_batch,
    team_shares,
)
from sindy_wm.models.switch_library import UNITS, scaled
from sindy_wm.paths import DATA_DIR

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
PROCESSED = DATA_DIR / "processed" / "exp06_intents"
COLORS = {
    "real battle": "black",
    "current model": "tab:blue",
    "generic SINDYc": "tab:orange",
    "Lanchester": "tab:green",
    "persistence": "tab:gray",
}

train = add_rates(add_controls(pd.read_parquet(PROCESSED / "train.parquet")))
val = add_controls(pd.read_parquet(PROCESSED / "val.parquet"))
rows = train[(train["alive_blue"] > 0) & (train["alive_red"] > 0)]
P_hand = pickle.load(open(PROCESSED / "closures.pkl", "rb"))
n_train, n_val = train.index.get_level_values(0).nunique(), val.index.get_level_values(0).nunique()
print(f"{n_train} training battles, {n_val} validation battles")

## 1 · The models

From simplest to the current one. All predict from a start state and the intent sequence Blue
actually played; none sees the future of the battle.

| model | state | control | what decides its terms |
|---|---|---|---|
| **persistence** | – | – | nothing changes (baseline) |
| **Lanchester** | health | none | the classic square law `H_B' = -a H_R`, `H_R' = -b H_B`: 🟡 form designed, a, b fitted by least squares |
| **generic SINDYc** | health, alive, distance | hold, withdraw, focus | 🟢 pure SINDy: degree-2 polynomials of state and control, nothing designed |
| **physics, no control** | health, alive, distance | none | 🟢 SINDy on a physics library (units in reach), intents ignored |
| **sparse SINDYc** (`model_sparse.pkl`) | same | one-hot intent | 🟢 SINDy on the physics library with control products, 🟡 hand-fitted reach curve |
| **+ hold split by contact** (`model_split.pkl`) | same | same | as above, hold's movement term split into before / in contact |
| **current model** (`model_derived.pkl`) | same | same | as above, with the reach curve 🔵 derived from the game's reach + 🟡 three fitted factors |

In [ ]:
# Lanchester: least squares on health only, no SINDy
both_alive = rows[["health_blue", "health_red"]].to_numpy()
a = -np.dot(both_alive[:, 1], rows["d_health_blue"]) / np.dot(both_alive[:, 1], both_alive[:, 1])
b = -np.dot(both_alive[:, 0], rows["d_health_red"]) / np.dot(both_alive[:, 0], both_alive[:, 0])
print(f"Lanchester: H_B' = -{a:.3f} H_R,  H_R' = -{b:.3f} H_B")

# generic SINDYc: pysindy, polynomial library of degree 2 on [state, hold, withdraw, focus]
generic = ps.SINDy(
    optimizer=ps.STLSQ(threshold=0.01, alpha=1e-3),
    feature_library=ps.PolynomialLibrary(degree=2),
    differentiation_method=ps.SmoothedFiniteDifference(smoother_kws={"window_length": 9, "polyorder": 2}),
)
groups = [g for _, g in train.groupby(level=0, sort=False)]
generic.fit(
    [g[STATE].to_numpy() for g in groups],
    t=DT,
    u=[g[CTRL[1:]].to_numpy() for g in groups],
    feature_names=STATE + CTRL[1:],
)
print(f"generic SINDYc: {int((generic.coefficients() != 0).sum())} terms in 5 equations")

models = {
    "physics, no control": IntentModel(P_hand, ctrl=False).fit(rows, thr_shape=None),
    "sparse SINDYc": pickle.load(open(PROCESSED / "model_sparse.pkl", "rb")),
    "+ hold split": pickle.load(open(PROCESSED / "model_split.pkl", "rb")),
    "current model": pickle.load(open(PROCESSED / "model_derived.pkl", "rb")),
}
current = models["current model"]
for name, m in models.items():
    print(f"\n== {name}")
    m.print(shape=False)

## 2 · Anatomy of the current model: what is SINDy and what is not

In [ ]:
fam = {
    "health_blue": ("Rb", "Rr", "Ab", "Ar"),
    "health_red": ("Rb", "Rr", "Ab", "Ar"),
    "distance": ("close", "flee"),
}
anatomy = []
for eq, family in fam.items():
    cands = [n for n in current.names if n.split("*")[0] in family and n not in current.exclude]
    kept = [n for n in cands if current.coef[eq][current.names.index(n)] != 0]
    anatomy.append(
        {
            "part": f"{eq}: which terms",
            "equation": format_terms(kept, [current.coef[eq][current.names.index(n)] for n in kept]),
            "label": "🟢 SINDy",
            "detail": f"{len(kept)} of {len(cands)} candidates kept: {', '.join(cands)}",
        }
    )
anatomy += [
    {
        "part": "units in reach R = A·s",
        "equation": "s = Φ((ρ − d_eff + β·depth_opp·√(2 ln n_opp)) / (g·depth_own))",
        "label": "🔵 game rule + 🟡 fitted outside SINDy",
        "detail": "ρ = 5.75 (game); λ, β, g per side: "
        + "; ".join(f"{s} {np.round(current.P[f'share_{s}'][:3], 2).tolist()}" for s in ("blue", "red")),
    },
    {
        "part": "switches close_*, flee",
        "equation": "Φ((d − c − b·depths − e·widths) / spread)",
        "label": "🟡 designed, fitted outside SINDy",
        "detail": "about 30 numbers in total (8 per closing switch, 4 for flee), from curve_fit",
    },
    {
        "part": "contact",
        "equation": "1 − (1 − s_B)^(10·A_B)",
        "label": "⚪ designed, no parameters",
        "detail": "chance that at least one Blue unit is in reach",
    },
    {
        "part": "units alive",
        "equation": "(ln A)' = p(u)·(ln H)'",
        "label": "⚪ form designed + 🟢 p by sparse regression",
        "detail": f"p_blue = {current.p['blue']['1']:.3f}; p_red = {current.p['red']['1']:.3f} "
        f"+ {current.p['red']['focus']:.3f}·focus (hold, withdraw candidates dropped)",
    },
    {
        "part": "formation (depth, width)",
        "equation": "held at the start value",
        "label": "⚪ designed, no parameters",
        "detail": "shape equations exist but fit with R² < 0.1",
    },
    {
        "part": "intent",
        "equation": "u = [advance, hold, withdraw, focus]",
        "label": "⚪ designed, no parameters",
        "detail": "one-hot: the options an LLM would choose between",
    },
    {
        "part": "health candidates themselves",
        "equation": "units in reach (Rb, Rr), alive (Ab, Ar) × {1, hold, wd, focus}",
        "label": "⚪ designed",
        "detail": "the library: SINDy can only choose among what it is offered",
    },
]
pd.DataFrame(anatomy).set_index("part")

**Count of numbers:** 🟢 SINDy coefficients 8 (3 health, 5 distance) + 3 alive exponents;
🟡 fitted outside SINDy: 6 reach-curve factors and about 30 switch parameters; 🔵 1 game constant (ρ).
Note that the distance library is small (5 candidates) and SINDy kept all five: there, the
"selection" is mostly done by how the library was designed.

## 2b · Is health selected out when it is offered?

The current library offers units in reach and units alive, but not health, as drivers of the
health equations: health was left out because it explained nothing in a separate test. Here SINDy
decides instead: 9 health candidates are added (Blue's and Red's health, each times every intent,
and health_blue × health_red, the Lanchester-type product) and the health equations are refitted at
three thresholds, once on the measured units in reach and once on the derived reach curve.

In [ ]:
from sindy_wm.models.intent_model import ALIVE_CONTROL_TERMS, equation, library, r2, stlsq  # noqa: E402

val_rows = add_rates(val)
val_rows = val_rows[(val_rows["alive_blue"] > 0) & (val_rows["alive_red"] > 0)]


def health_library(df, measured_reach):
    X, U, S = df[STATE].to_numpy(), df[CTRL].to_numpy(), df[SHAPE].to_numpy()
    F = library(X, U, S, current.P, df[REACH_COLS].to_numpy() if measured_reach else None)
    F = {
        n: v
        for n, v in F.items()
        if n.split("*")[0] in ("Rb", "Rr", "Ab", "Ar") and n not in ALIVE_CONTROL_TERMS
    }
    for suffix, c in (("", 1), ("*hold", U[:, 1]), ("*wd", U[:, 2]), ("*focus", U[:, 3])):
        F["Hb" + suffix], F["Hr" + suffix] = X[:, 0] * c, X[:, 1] * c  # Hb, Hr: Blue's and Red's health
    F["Hb*Hr"] = X[:, 0] * X[:, 1]
    return list(F), np.column_stack(list(F.values()))


selection = []
for measured in (True, False):
    names, T = health_library(rows, measured)
    _, Tv = health_library(val_rows, measured)
    for thr in (0.05, 0.1, 0.2):
        for eq in ("health_blue", "health_red"):
            c = stlsq(T, rows["d_" + eq].to_numpy(), thr)
            kept = [(n, v) for n, v in zip(names, c, strict=True) if v]
            selection.append(
                {
                    "units in reach": "measured" if measured else "derived curve",
                    "threshold": thr,
                    "equation": eq,
                    "kept terms": format_terms([n for n, _ in kept], [v for _, v in kept]),
                    "health terms kept": sum(n.startswith("H") for n, _ in kept),
                    "R² (validation)": r2(val_rows["d_" + eq].to_numpy(), Tv @ c),
                }
            )
print(f"{len(names)} candidates per equation, of which 9 are health terms")
pd.DataFrame(selection).set_index(["units in reach", "threshold", "equation"]).round(3)

**What the numbers say:** fill in. Questions to ask: does the *opponent's* health ever enter as a
driver? At the threshold the current model uses (0.2), what survives? When a health term survives,
whose health is it, and what could it stand in for? What happens to the withdraw term (`Rb*wd`)
when `Hb*wd` is also on offer?

## 3 · Whole battles: model against the real battle

Four validation battles. Each model starts from the real state 1 s in and is given the intents Blue
actually played; from there it predicts the whole battle on its own. Background colour: Blue's
movement (green advance, blue hold, red withdraw).

In [ ]:
MOVE_COLOR = {"advance": "#d9f2d9", "hold": "#d9e6f7", "withdraw": "#f7dada"}


def rollout(name, X0, S0, R0, U):
    """(N, steps+1, 5) prediction of one model from start states, given the intents U (N, steps, 4)."""
    steps = U.shape[1]
    if name == "persistence":
        return np.repeat(X0[:, None], steps + 1, 1)
    if name == "Lanchester":
        X, out = X0.astype(float).copy(), [X0.astype(float).copy()]
        for _ in range(steps):
            X = X.copy()
            X[:, :2] = np.maximum(X[:, :2] + DT * np.column_stack([-a * X[:, 1], -b * X[:, 0]]), 0)
            out.append(X)
        return np.stack(out, 1)
    if name == "generic SINDYc":
        X, out = X0.astype(float).copy(), [X0.astype(float).copy()]
        for n in range(steps):
            X = np.clip(
                X + DT * generic.predict(X, u=U[:, n, 1:]), 0, 30
            )  # clipped: polynomials can run away
            out.append(X)
        return np.stack(out, 1)
    return models[name].simulate(X0, S0, U, reach0=R0)


ep_info = val.groupby(level=0).agg(
    won=("battle_won", "first"), seconds=("t", "max"), withdraw=("withdraw", "mean")
)
ep_info["final margin"] = val.groupby(level=0).apply(
    lambda g: g["health_blue"].iloc[-1] - g["health_red"].iloc[-1]
)
long = ep_info[ep_info["seconds"] > 8]
picks = {
    "Blue wins (typical battle)": (wins := long[long["won"]]["seconds"]).sub(wins.median()).abs().idxmin(),
    "Red wins": long[~long["won"]]["final margin"].idxmin(),
    "much withdrawing": long["withdraw"].idxmax(),
    "close fight": long["final margin"].abs().idxmin(),
}
SHOW = ["real battle", "current model", "generic SINDYc", "Lanchester"]


def battle(ep, k0=8):
    g = val.loc[ep]
    return g, g[STATE].to_numpy(), g[SHAPE].to_numpy(), g[REACH_COLS].to_numpy(), g[CTRL].to_numpy()


fig, axes = plt.subplots(len(picks), 4, figsize=(17, 3.2 * len(picks)))
for row, (title, ep) in zip(axes, picks.items(), strict=True):
    g, X, S, R, U = battle(ep)
    k0, t = 8, g["t"].to_numpy()
    U_rest = U[k0:-1][None]
    preds = {name: rollout(name, X[k0][None], S[k0][None], R[k0][None], U_rest)[0] for name in SHOW[1:]}
    reach_model = preds["current model"][:, 2:4] * np.column_stack(
        team_shares(preds["current model"], np.repeat(S[k0][None], len(preds["current model"]), 0), current.P)
    )
    panels = [("health", [0, 1]), ("units alive", [2, 3]), ("distance", [4]), ("units in reach", None)]
    for ax, (label, cols) in zip(row, panels, strict=True):
        for mv, c in MOVE_COLOR.items():
            ax.fill_between(
                t,
                0,
                1,
                where=g["movement"].to_numpy() == mv,
                color=c,
                transform=ax.get_xaxis_transform(),
                lw=0,
            )
        ls = {0: "-", 1: "--", 2: "-", 3: "--", 4: "-"}
        if cols is None:
            for j, side in enumerate(("Blue", "Red")):
                ax.plot(t, 10 * R[:, j], color="black", ls=["-", "--"][j], label=f"real ({side})")
                ax.plot(t[k0:], 10 * reach_model[:, j], color=COLORS["current model"], ls=["-", "--"][j])
        else:
            for c in cols:
                ax.plot(t, X[:, c] * (10 if label == "units alive" else 1), color="black", ls=ls[c])
                for name in SHOW[1:]:
                    if label != "health" and name == "Lanchester":
                        continue  # Lanchester only models health
                    ax.plot(
                        t[k0:],
                        preds[name][:, c] * (10 if label == "units alive" else 1),
                        color=COLORS[name],
                        ls=ls[c],
                        alpha=0.9,
                    )
        ax.axvline(t[k0], color="gray", lw=0.8, ls=":")
        ax.set(title=f"{title}: {label}", xlabel="seconds")
handles = [plt.Line2D([], [], color=COLORS[n], label=n) for n in SHOW] + [
    plt.Line2D([], [], color="gray", ls="-", label="solid: Blue"),
    plt.Line2D([], [], color="gray", ls="--", label="dashed: Red"),
]
fig.legend(handles=handles, loc="upper center", ncol=6)
fig.tight_layout(rect=(0, 0, 1, 0.97))

**What the plots show:** fill in. Questions to ask: where does each model leave the real battle
first? Do the models get the *timing* of contact right (distance)? Does the current model's
"units in reach" follow the real one, and where not?

## 4 · Rolling forecasts: what the planner sees

A planner doesn't predict a whole battle once; it predicts a few seconds ahead, again and again.
Here: every 1 s, a 4 s forecast of the health margin (Blue − Red) by the current model, against the
real margin.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
for ax, (title, ep) in zip(axes, list(picks.items())[:2], strict=True):
    g, X, S, R, U = battle(ep)
    t, margin = g["t"].to_numpy(), X[:, 0] - X[:, 1]
    ax.plot(t, margin, color="black", lw=2, label="real margin")
    for k0 in range(8, len(g) - 4, 8):
        steps = min(32, len(g) - 1 - k0)
        Y = rollout("current model", X[k0][None], S[k0][None], R[k0][None], U[k0 : k0 + steps][None])[0]
        ax.plot(
            t[k0 : k0 + steps + 1],
            Y[:, 0] - Y[:, 1],
            color=COLORS["current model"],
            alpha=0.7,
            label="4 s forecast (current model)" if k0 == 8 else None,
        )
        ax.plot(t[k0], margin[k0], "o", color=COLORS["current model"], ms=3)
    ax.axhline(0, color="gray", lw=0.5)
    ax.set(title=f"{title}: 4 s forecasts every 1 s", xlabel="seconds", ylabel="health margin, Blue − Red")
axes[0].legend()

## 5 · Errors against horizon, with bootstrap intervals

Start points every 1 s in all validation battles; real intents applied. Mean absolute error of
health (share of ten full units) and distance. The 95% interval resamples whole **battles** (2,000
times): how much the number could change with another set of battles.

In [ ]:
B = make_batch(val)
all_models = ["persistence", "Lanchester", "generic SINDYc", *models]
preds = {name: rollout(name, B["X0"], B["shape0"], B["reach0"], B["U"]) for name in all_models}
table = pd.DataFrame({name: horizon_errors(p, B) for name, p in preds.items()}).T

rng = np.random.default_rng(0)
episodes, inverse = np.unique(B["ep"], return_inverse=True)
n4 = round(4 / DT)
ok = ~np.isnan(B["Y"][:, n4, 0])
idx = rng.integers(0, len(episodes), (2000, len(episodes)))
for name, p in preds.items():
    err = np.where(ok, np.abs(p[:, n4, :2] - B["Y"][:, n4, :2]).mean(1), 0)
    s, c = np.bincount(inverse, err, len(episodes)), np.bincount(inverse, ok, len(episodes))
    boot = s[idx].sum(1) / c[idx].sum(1)
    table.loc[name, "health 4 s, 95% interval"] = (
        f"{np.quantile(boot, 0.025):.3f} – {np.quantile(boot, 0.975):.3f}"
    )
display(table.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
line_color = dict(zip(all_models, plt.cm.tab10.colors, strict=False))
for name in all_models:
    h = [1, 2, 4, 8]
    for j, q in enumerate(("health", "distance")):
        if q == "distance" and name == "Lanchester":
            continue
        ax[j].plot(h, [table.loc[name, f"{q} {x} s"] for x in h], "o-", label=name, color=line_color[name])
        ax[j].set(xlabel="horizon (s)", ylabel=f"mean absolute error, {q}", xticks=h)
ax[0].legend(fontsize=8)
fig.tight_layout()

## 5b · Predictions over horizons

The same start points as section 5 (every 1 s in every validation battle). Three views:

1. **Predicted against real change** in the health margin (Blue − Red) over 1, 2, 4 and 8 s, for
   the current model. The *change* is used, not the level: the level is easy because it starts
   right.
2. **How the error grows** with the horizon, every 0.5 s up to 8 s: median and the 10-90% band of
   the absolute margin error, for the current model, generic SINDYc and persistence.
3. **Where the error comes from:** error at 4 s split by the situation at the start (apart or in
   contact) and by what Blue does over those 4 s.

In [ ]:
margin_real = B["Y"][:, :, 0] - B["Y"][:, :, 1]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
pc = preds["current model"]
for ax, h in zip(axes, (1, 2, 4, 8), strict=True):
    n = round(h / DT)
    okh = ~np.isnan(margin_real[:, n])
    real_change = margin_real[okh, n] - margin_real[okh, 0]
    pred_change = (pc[okh, n, 0] - pc[okh, n, 1]) - (pc[okh, 0, 0] - pc[okh, 0, 1])
    ax.scatter(real_change, pred_change, s=2, alpha=0.25, color=COLORS["current model"])
    lim = np.quantile(np.abs(np.r_[real_change, pred_change]), 0.995)
    ax.plot([-lim, lim], [-lim, lim], color="black", lw=0.8)
    r2h = 1 - np.sum((pred_change - real_change) ** 2) / np.sum((real_change - real_change.mean()) ** 2)
    ax.set(
        xlim=(-lim, lim),
        ylim=(-lim, lim),
        title=f"{h} s ahead: R² {r2h:.2f}",
        xlabel="real change in margin",
        ylabel="predicted change" if h == 1 else None,
    )
fig.suptitle("current model: predicted vs real change in health margin (Blue − Red)")
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
steps_h = np.arange(4, 65, 4)
for name in ("persistence", "generic SINDYc", "current model"):
    p = preds[name]
    err = np.abs((p[:, :, 0] - p[:, :, 1]) - margin_real)
    q = np.array([np.nanquantile(err[:, n], [0.1, 0.5, 0.9]) for n in steps_h])
    ax.plot(steps_h * DT, q[:, 1], color=COLORS[name], label=f"{name} (median)")
    ax.fill_between(steps_h * DT, q[:, 0], q[:, 2], color=COLORS[name], alpha=0.15)
ax.set(
    xlabel="horizon (s)",
    ylabel="absolute error in health margin",
    title="error against horizon: median and 10-90% band",
)
ax.legend()

In [ ]:
in_contact = B["reach0"].sum(axis=1) > 0
moves = np.array(["advance", "hold", "withdraw"])
main_move = moves[B["U"][:, :32, :3].mean(axis=1).argmax(axis=1)]
pure = B["U"][:, :32, :3].mean(axis=1).max(axis=1) > 0.99  # one movement over all 4 s
n4 = round(4 / DT)
ok4 = ~np.isnan(margin_real[:, n4])
rows_ = []
for name in ("persistence", "generic SINDYc", "sparse SINDYc", "current model"):
    p = preds[name]
    err = np.abs((p[:, n4, 0] - p[:, n4, 1]) - margin_real[:, n4])
    for where, mask in (("apart at the start", ~in_contact), ("in contact at the start", in_contact)):
        for mv in moves:
            sel = ok4 & mask & pure & (main_move == mv)
            rows_.append(
                {
                    "model": name,
                    "start": where,
                    "Blue's movement for 4 s": mv,
                    "error at 4 s": err[sel].mean(),
                    "start points": int(sel.sum()),
                }
            )
by_situation = pd.DataFrame(rows_)
display(
    by_situation.pivot_table(
        index=["start", "Blue's movement for 4 s"], columns="model", values="error at 4 s"
    )
    .round(3)
    .join(by_situation.groupby(["start", "Blue's movement for 4 s"])["start points"].first())
)

**What the numbers say:** fill in. Questions to ask: up to which horizon does the model predict
the *change* in margin well? Does its error band stay narrower than generic SINDYc's? In which
situation is it worst, and does that match the battles in section 3?

## 6 · Where it goes wrong: units in reach

The health equations multiply units in reach by a constant, so most of the error comes from
getting units in reach wrong. Here the reach curves are given the **real** distance and formation
at every step, so only the curve itself can be wrong.

In [ ]:
from sindy_wm.models.intent_model import share  # noqa: E402

fig, axes = plt.subplots(1, 4, figsize=(17, 3.2), sharey=True)
for ax, (title, ep) in zip(axes, picks.items(), strict=True):
    g, X, S, R, U = battle(ep)
    t = g["t"].to_numpy()
    hand = X[:, 2] * share(X[:, 4], S[:, 0], S[:, 1], S[:, 2], S[:, 3], *P_hand["share_blue"])
    derived = X[:, 2] * team_shares(X, S, current.P)[0]
    ax.plot(t, 10 * R[:, 0], color="black", lw=2, label="real")
    ax.plot(t, 10 * derived, color=COLORS["current model"], label="derived curve (current)")
    ax.plot(t, 10 * hand, color="tab:purple", ls=":", lw=1.8, label="hand-fitted curve")
    ax.set(title=f"{title}: Blue units in reach", xlabel="seconds")
axes[0].legend(fontsize=8)
fig.tight_layout()

## 6b · Is the missing piece the formation, or the distance?

During a prediction the formation (depth, width) is normally **held** at its start value. Here the
model gets, over the whole horizon, the **real** formation, the **real** distance, or both (taken
from the battle), against the default and against the formation's own equations. Errors in units
in reach (count of units), health and health margin; then the hold-vs-advance pairs from
`sindyc_05` (12 s ahead: does the model predict that holding wins?).

In [ ]:
shape_variants = {
    "formation held (default)": {"shape": "hold"},
    "formation from its equations": {"shape": "model"},
    "real formation": {"truth": {"shape": B["Yshape"]}},
    "real distance": {"truth": {"distance": B["Y"][:, :, 4]}},
    "real distance + real formation": {"truth": {"distance": B["Y"][:, :, 4], "shape": B["Yshape"]}},
}
shape_rows = []
for name, kw in shape_variants.items():
    Y, Sh = current.simulate(B["X0"], B["shape0"], B["U"], reach0=B["reach0"], return_shape=True, **kw)
    if "shape" in kw.get("truth", {}):
        Sh = np.where(np.isnan(B["Yshape"]), Sh, B["Yshape"])
    rec = {"variant": name}
    for h in (1, 4, 8):
        n = round(h / DT)
        okh = ~np.isnan(B["Y"][:, n, 0]) & ~np.isnan(B["Yreach"][:, n, 0])
        sb, sr = team_shares(Y[okh, n], Sh[okh, n], current.P)
        R_hat = np.column_stack([Y[okh, n, 2] * sb, Y[okh, n, 3] * sr])
        rec[f"units in reach {h} s"] = 10 * np.mean(np.abs(R_hat - B["Yreach"][okh, n]))
        rec[f"health {h} s"] = np.mean(np.abs(Y[okh, n, :2] - B["Y"][okh, n, :2]))
        rec[f"margin {h} s"] = np.mean(np.abs((Y[okh, n, 0] - Y[okh, n, 1]) - margin_real[okh, n]))
    shape_rows.append(rec)
display(pd.DataFrame(shape_rows).set_index("variant").round(3))

pairs = pd.read_parquet(PROCESSED / "pairs_hold_advance.parquet").sort_values(["seed", "movement", "step"])
H12, pair_seeds = 96, np.sort(pairs["seed"].unique())


def pair_traj(movement, cols):
    g = pairs[pairs["movement"] == movement]
    out = np.full((len(pair_seeds), H12 + 1, len(cols)), np.nan)
    for j, (_, e) in enumerate(g.groupby("seed")):
        n = min(len(e), H12 + 1)
        out[j, :n] = e[cols].to_numpy()[:n]
    return out


last_rows = {
    mv: pairs[pairs["movement"] == mv].groupby("seed").last().loc[pair_seeds] for mv in ("hold", "advance")
}
real_diff = (last_rows["hold"]["health_blue"] - last_rows["hold"]["health_red"]).values - (
    last_rows["advance"]["health_blue"] - last_rows["advance"]["health_red"]
).values
pair_differs = np.abs(real_diff) > 0.02
PX0, PS0, PR0 = (pair_traj("hold", c)[:, 0] for c in (STATE, SHAPE, REACH_COLS))
decision = {}
for name, kind in (
    ("formation held (default)", None),
    ("formation from its equations", "model"),
    ("real formation", "shape"),
    ("real distance + real formation", "both"),
):
    p = {}
    for mv in ("hold", "advance"):
        U = np.repeat(np.repeat(control_vector(mv, "focus")[None], H12, 0)[None], len(pair_seeds), 0)
        kw = {"shape": "model"} if kind == "model" else {}
        if kind in ("shape", "both"):
            kw["truth"] = {"shape": pair_traj(mv, SHAPE)}
        if kind == "both":
            kw["truth"]["distance"] = pair_traj(mv, ["distance"])[:, :, 0]
        Y = current.simulate(PX0, PS0, U, reach0=PR0, **kw)
        p[mv] = Y[:, -1, 0] - Y[:, -1, 1]
    dd = p["hold"] - p["advance"]
    decision[name] = {
        "predicted hold − advance": dd.mean(),
        "real": real_diff.mean(),
        "correlation with real": np.corrcoef(dd, real_diff)[0, 1],
        "sign right where they differ": np.mean(
            np.sign(dd[pair_differs]) == np.sign(real_diff[pair_differs])
        ),
    }
pd.DataFrame(decision).T.round(3)

**What the numbers say:** fill in. Questions to ask: does the real formation on its own help?
Does it help once the distance is right too? Which matters more for units in reach: the formation
or the distance? What does that say about where to improve the model first?

## 6c · Can the distance law be derived from the game's movement rules?

SMAClite's rules: a unit walks (at 3.15) while it has no enemy in reach, then stops and shoots.
Red always does this; Blue does it under advance, under hold only once in contact, and under
withdraw walks away from enemies closer than 7. If the team centres moved like their walking units,

d' = −3.15 · (1 − s_R) − 3.15 · (1 − s_B) · (advance + hold · contact) + 3.15 · danger_B · withdraw

with s the reach curves (danger: the same curve at radius 7): no switch shapes to fit. Below: the
real closing speed against this rule, binned by how many units are walking; then SINDy on the
rule's terms (`RULE_MOVE_FAMILY`) against the hand-fitted switches.

In [ ]:
from sindy_wm.models.intent_model import RULE_MOVE_FAMILY  # noqa: E402

walking = val_rows.assign(
    walk_r=1 - val_rows["reach_red"] / val_rows["alive_red"],
    walk_b=1 - val_rows["reach_blue"] / val_rows["alive_blue"],
)
rule_check = []
for mv, blue_walks in (("advance", 1), ("hold", 0)):
    a = walking[walking["movement"] == mv]
    w = a["walk_r"] + blue_walks * a["walk_b"]
    bins = pd.cut(w, [-0.01, 0.05, 0.3, 0.6, 0.9, 1.2, 1.5, 1.8, 2.01]).rename(
        "teams' worth of units walking"
    )
    g = a.assign(w=w).groupby(bins, observed=True)
    t = g.agg(real_closing=("d_distance", "mean"), steps=("w", "size"))
    t["rule: −3.15 × walking"] = -3.15 * g["w"].mean()
    rule_check.append(t.assign(movement=mv).reset_index())
display(pd.concat(rule_check).set_index(["movement", "teams' worth of units walking"]).round(2))

rule_model = IntentModel(current.P, move_family=RULE_MOVE_FAMILY, exclude=current.exclude)
rule_model.fit(rows, thr_health=0.2, thr_distance=0.05, reach_true=True)
print("SINDy on the rule terms:")
print(
    "  "
    + equation(rule_model.names, rule_model.coef["distance"], "distance")
    + f"   [R² {rule_model.fit_r2['distance']:.2f}]"
)
print(f"  hand-fitted switches: R² {current.fit_r2['distance']:.2f}")
display(
    pd.DataFrame(
        {
            "hand-fitted switches (current)": horizon_errors(preds["current model"], B),
            "rule terms, SINDy coefficients": horizon_errors(
                rule_model.simulate(B["X0"], B["shape0"], B["U"], reach0=B["reach0"]), B
            ),
        }
    ).T.round(3)
)

**What the numbers say:** fill in. Questions to ask: where does the real closing speed fall short
of the rule? What in the game could stop units that are "walking" from moving the team centre
forward (think of units behind their own front line)? What would a derivation need to include?

## 7 · Counterfactuals: the same state, six futures

From one state of the counterfactual set, the real game was played with each intent for 4 s (then
advance/nearest). Solid: the real futures. Dashed: the current model's prediction for each.
The state is in contact, where holding the line *is* advancing: the advance lines lie under the
hold lines (real and predicted).

In [ ]:
cf = {i: pd.read_parquet(PROCESSED / f"cf_{i[0]}_{i[1]}.parquet") for i in INTENTS}
spread = {}
for ep, g in cf[INTENTS[0]].groupby(level=0):
    k = int(round(g["branch_at"].iloc[0] / DT))
    if 8 <= k < len(g) - 40 and g["alive_red"].iloc[k] > 0:
        finals = [
            cf[i].loc[ep]["health_blue"].iloc[-1] - cf[i].loc[ep]["health_red"].iloc[-1] for i in INTENTS
        ]
        spread[ep] = (max(finals) - min(finals), k)
ep, (_, k) = max(spread.items(), key=lambda kv: kv[1][0])
fig, ax = plt.subplots(figsize=(9, 4))
colors = dict(zip(INTENTS, plt.cm.tab10.colors, strict=False))
for i in INTENTS:
    if i == ("withdraw", "focus"):
        continue  # identical to withdraw/nearest
    g = cf[i].loc[ep]
    t = g["t"].to_numpy()
    ax.plot(t, g["health_blue"] - g["health_red"], color=colors[i], label=f"{i[0]}/{i[1]}")
    steps = len(g) - 1 - k
    U = np.repeat(control_vector("advance", "nearest")[None], steps, 0)
    U[:32] = control_vector(*i)
    X0, S0, R0 = (g[c].to_numpy()[k][None] for c in (STATE, SHAPE, REACH_COLS))
    Y = current.simulate(X0, S0, U[None], reach0=R0)[0]
    ax.plot(t[k:], Y[:, 0] - Y[:, 1], color=colors[i], ls="--")
ax.axvline(t[k], color="gray", ls=":")
ax.axvspan(t[k], t[k] + 4, color="gray", alpha=0.1)
ax.axhline(0, color="gray", lw=0.5)
ax.set(
    xlabel="seconds",
    ylabel="health margin, Blue − Red",
    title="one state, each intent for 4 s (shaded): real (solid) vs model (dashed)",
)
ax.legend(fontsize=8)

## 8 · Transfer: a stalker battle, predicted by the marine model

The current model fitted on marines, used on stalkers (exp07) without refitting: **as is**, and
**scaled** by the stalker's reach, time to kill and walking speed (🔵 game constants only).
Background colour as in section 3: Blue's movement (green advance, blue hold, red withdraw).

In [ ]:
s_val = add_controls(pd.read_parquet(DATA_DIR / "processed" / "exp07_stalkers" / "val.parquet"))
s_info = s_val.groupby(level=0)["t"].max()
s_ep = s_info[s_info > 15].index[0]
g = s_val.loc[s_ep]
X, S, R, U = (g[c].to_numpy() for c in (STATE, SHAPE, REACH_COLS, CTRL))
t, k0 = g["t"].to_numpy(), 8
variants = {
    "marine model, as is": current,
    "marine model, scaled to stalkers": scaled(current, "marine", "stalker"),
}
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, (label, cols) in zip(
    axes, [("health", [0, 1]), ("units alive", [2, 3]), ("distance", [4])], strict=True
):
    for mv, c in MOVE_COLOR.items():
        ax.fill_between(
            t, 0, 1, where=g["movement"].to_numpy() == mv, color=c, transform=ax.get_xaxis_transform(), lw=0
        )
    for c in cols:
        f = 10 if label == "units alive" else 1
        ax.plot(t, X[:, c] * f, color="black", ls="-" if c in (0, 2, 4) else "--")
        for m, color in zip(variants.values(), ("tab:red", "tab:blue"), strict=True):
            Y = m.simulate(X[k0][None], S[k0][None], U[k0:-1][None], reach0=R[k0][None])[0]
            ax.plot(t[k0:], Y[:, c] * f, color=color, ls="-" if c in (0, 2, 4) else "--")
    ax.set(title=f"stalkers: {label}", xlabel="seconds")
fig.legend(
    handles=[
        plt.Line2D([], [], color="black", label="real battle"),
        plt.Line2D([], [], color="tab:red", label="marine model, as is"),
        plt.Line2D([], [], color="tab:blue", label="marine model, scaled (no refit)"),
        plt.Line2D([], [], color="gray", ls="-", label="solid: Blue"),
        plt.Line2D([], [], color="gray", ls="--", label="dashed: Red"),
    ],
    loc="upper center",
    ncol=5,
    bbox_to_anchor=(0.5, 1.05),
)
fig.tight_layout()
print(
    "scaling (🔵 game constants): reach",
    UNITS["marine"]["reach"],
    "→",
    UNITS["stalker"]["reach"],
    "; time to kill",
    round(UNITS["marine"]["tau"], 2),
    "→",
    round(UNITS["stalker"]["tau"], 2),
    "s; speed",
    UNITS["marine"]["speed"],
    "→",
    UNITS["stalker"]["speed"],
)

## 9 · Choosing the reach factors by how well the health equations fit

The reach curve's factors λ, β, g (🟡) are fitted to the **measured** units in reach. An
alternative that needs no measured reach: choose the factors that make the **health equations**
fit best (for each side, the shooter's curve against the target's health loss, with the
coefficients solved by least squares inside the search). Then refit the model with SINDy on that
curve and compare: the factors, what the curve still means, prediction, and the planner.

In [ ]:
from scipy.optimize import minimize  # noqa: E402

from sindy_wm.models.intent_model import derived_share  # noqa: E402

fit_rows = rows.sample(200_000, random_state=0)


def fire_parts(df, side):
    o = "red" if side == "blue" else "blue"
    X = (
        df["distance"].values,
        df[f"depth_{side}"].values,
        df[f"depth_{o}"].values,
        df[f"width_{side}"].values,
        df[f"width_{o}"].values,
        10 * df[f"alive_{o}"].values,
    )
    extra = df["withdraw"].values if side == "blue" else None  # Blue's fire has the withdraw product
    return X, df[f"alive_{side}"].values, df[f"d_health_{o}"].values, extra


def fire_design(params, X, A, extra):
    R = A * derived_share(*X, *params)
    return np.column_stack([R, R * extra]) if extra is not None else R[:, None]


def health_loss(params, X, A, y, extra):
    if np.any(np.asarray(params) <= 0):
        return 1e9
    T = fire_design(params, X, A, extra)
    return np.mean((y - T @ np.linalg.lstsq(T, y, rcond=None)[0]) ** 2)


P_health = dict(current.P)
factor_table = {}
for side in ("blue", "red"):
    X, A_, y, extra = fire_parts(fit_rows, side)
    fits = [
        minimize(
            health_loss,
            x0,
            args=(X, A_, y, extra),
            method="Nelder-Mead",
            options={"xatol": 1e-3, "fatol": 1e-9, "maxiter": 400},
        )
        for x0 in ([1.0, 1.0, 1.0], [1.0, 0.5, 1.5], [2.0, 0.7, 0.8])
    ]
    p_h = min(fits, key=lambda r: r.fun).x
    P_health[f"share_{side}"] = np.append(p_h, current.P[f"share_{side}"][3])
    Xv, Av, yv, ev = fire_parts(val_rows, side)
    for label, pp in (
        ("from measured reach (current)", current.P[f"share_{side}"][:3]),
        ("from the health fit", p_h),
    ):
        coef = np.linalg.lstsq(fire_design(pp, X, A_, extra), y, rcond=None)[0]
        factor_table[(side, label)] = {
            "λ, β, g": np.round(pp, 2),
            "health equation R² (val)": r2(yv, fire_design(pp, Xv, Av, ev) @ coef),
            "units in reach R² (val)": r2(val_rows[f"reach_{side}"].values, Av * derived_share(*Xv, *pp)),
        }
display(pd.DataFrame(factor_table).T.round(3))

health_fit_model = IntentModel(P_health, exclude=current.exclude).fit(rows, thr_health=0.2)
print("SINDy on the health-fitted curve:")
health_fit_model.print(shape=False)
Y_h = health_fit_model.simulate(B["X0"], B["shape0"], B["U"], reach0=B["reach0"])
compare_h = pd.DataFrame(
    {
        "current model": table.loc["current model", [c for c in table.columns if "interval" not in c]],
        "health-fitted curve": horizon_errors(Y_h, B),
    }
).T
display(compare_h.round(3))

**In the planner** (400 battles, `scripts/eval_planners.py`; computed outside this notebook with the
same model saved as `model_healthfit.pkl`):

| | current model | health-fitted curve |
|---|---|---|
| Blue wins | 69.8% | **11.0%** |
| chooses withdraw | 5% | **50%** |

**What the numbers say:** fill in. Questions to ask: what does the new health equation for Blue
say physically? How can two models with the same prediction error plan so differently? What does
that mean for "the factors can be found from health alone"?

## 10 · Figures for the meeting brief

The three figure sets of the supervisor-meeting brief, saved as PDF and PNG in `docs/figures/` for
Overleaf. Health is on a 0–1 scale (1 = a full team of ten full-health marines); units are counted
(0–14). All use the current model on the exp06 validation battles, with Blue's real intents.

1. `horizon_errors`: average error 0.5–8 s ahead over all 8,946 start points (every second of 1,000
   battles), for the current model, generic SINDy and "nothing changes".
2. `example_predictions`: three moments, 8 s ahead. All start points with at least 8 s of battle
   left are ranked by the model's health error after 4 s (both teams); the moments at the best 25%,
   the median and the worst 10% are shown. Solid: real; dashed: model.
3. `rolling_forecasts`: three whole battles with a 4 s forecast from the real state every second.
   Battles with at least three full forecasts are ranked by their average health error after 4 s;
   the best 25%, the median and the worst 10% are shown.

In [ ]:
from pathlib import Path  # noqa: E402

FIG_DIR = Path(DATA_DIR).parent / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TEAM = {"blue": "#2f5bd3", "red": "#c0392b"}
LINE = {
    "current model": ("#2f5bd3", "-", "Our model (8 terms)"),
    "generic SINDYc": ("#c27a12", "-", "Generic SINDy (138 terms)"),
    "persistence": ("#8a919e", "--", "No model: nothing changes"),
}
RANKS = (
    ("Small error at 4 s (best 25%)", 0.25),
    ("Typical error at 4 s (median)", 0.5),
    ("Large error at 4 s (worst 10%)", 0.9),
)
plt.rcParams.update({"font.size": 10, "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 9})


def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=200)
    print("saved", FIG_DIR / f"{name}.pdf")


def model_reach(Ypred, shape0, reach0):
    """Units in reach as the model computes it: reach curve with the start formation held, and the
    measured start reach fading into the curve with a time constant of 1 s."""
    n = Ypred.shape[1]
    S_ = np.repeat(shape0[:, None], n, 1).reshape(-1, 4)
    sb, sr = team_shares(Ypred.reshape(-1, 5), S_, current.P)
    s = np.column_stack([sb, sr]).reshape(len(Ypred), n, 2)
    offset = reach0 / np.maximum(Ypred[:, 0, 2:4], 1e-9) - s[:, 0]
    fade = np.exp(-np.arange(n) * DT / 1.0)[None, :, None]
    return Ypred[:, :, 2:4] * np.clip(s + offset[:, None] * fade, 0, 1)


# 1 · error against horizon
hs = np.arange(0.5, 8.01, 0.5)
R_cur = model_reach(preds["current model"], B["shape0"], B["reach0"])
curves = {}
for name in LINE:
    P_ = preds[name]
    c = {"health": [], "units alive": [], "distance": [], "units in reach": []}
    for h in hs:
        n = round(h / DT)
        okh = ~np.isnan(B["Y"][:, n, 0]) & ~np.isnan(B["Yreach"][:, n, 0])
        c["health"].append(np.mean(np.abs(P_[okh, n, :2] - B["Y"][okh, n, :2])))
        c["units alive"].append(10 * np.mean(np.abs(P_[okh, n, 2:4] - B["Y"][okh, n, 2:4])))
        c["distance"].append(np.mean(np.abs(P_[okh, n, 4] - B["Y"][okh, n, 4])))
        R_ = (
            R_cur[okh, n]
            if name == "current model"
            else (B["reach0"][okh] if name == "persistence" else None)
        )
        c["units in reach"].append(np.nan if R_ is None else 10 * np.mean(np.abs(R_ - B["Yreach"][okh, n])))
    curves[name] = c
panels = [
    ("health", "Health", "error (1 = full team)"),
    ("units alive", "Units alive", "error (units)"),
    ("distance", "Distance between the teams", "error (map units)"),
    ("units in reach", "Units in reach", "error (units)"),
]
fig, axes = plt.subplots(2, 2, figsize=(9, 6.2))
for ax, (key, title, ylab) in zip(axes.flat, panels, strict=True):
    for name, (col, ls, label) in LINE.items():
        if not np.all(np.isnan(curves[name][key])):
            ax.plot(
                hs,
                curves[name][key],
                color=col,
                ls=ls,
                lw=2.2 if name == "current model" else 1.6,
                label=label,
            )
    ax.axvline(4, color="0.8", ls=":", lw=1)
    ax.set(
        title=title, xlabel="seconds ahead", ylabel=ylab, xlim=(0, 8), ylim=(0, None), xticks=[0, 2, 4, 6, 8]
    )
    ax.grid(alpha=0.25)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.03))
fig.tight_layout()
save(fig, "horizon_errors")

In [ ]:
# 2 · three example predictions, 8 s ahead
Nh = B["U"].shape[1]
full = ~np.isnan(B["Y"][:, Nh, 0])
err4 = np.abs(preds["current model"][:, n4, :2] - B["Y"][:, n4, :2]).mean(1)
cand = np.flatnonzero(full)
ranked = cand[np.argsort(err4[cand])]
ex_picks = [(title, ranked[int(q * len(ranked))]) for title, q in RANKS]
print(f"{len(ranked)} start points with at least 8 s of battle left")
t8 = np.arange(Nh + 1) * DT
rows_spec = [
    ("Health (1 = full team)", lambda i, P_: (B["Y"][i, :, :2], P_[i, :, :2])),
    ("Units alive", lambda i, P_: (10 * B["Y"][i, :, 2:4], 10 * P_[i, :, 2:4])),
    ("Distance (map units)", lambda i, P_: (B["Y"][i, :, 4:5], P_[i, :, 4:5])),
    ("Units in reach", lambda i, P_: (10 * B["Yreach"][i], 10 * R_cur[i])),
]
fig, axes = plt.subplots(4, 3, figsize=(10, 10), sharex=True)
for c_, (title, i) in enumerate(ex_picks):
    print(f"{title}: health error after 4 s {err4[i]:.3f}")
    for r_, (ylab, f) in enumerate(rows_spec):
        ax = axes[r_, c_]
        real, pred = f(i, preds["current model"])
        if real.shape[1] == 1:
            ax.plot(t8, real[:, 0], color="black", lw=1.8)
            ax.plot(t8, pred[:, 0], color="black", lw=1.8, ls="--")
        else:
            for j, team in enumerate(("blue", "red")):
                ax.plot(t8, real[:, j], color=TEAM[team], lw=1.8)
                ax.plot(t8, pred[:, j], color=TEAM[team], lw=1.8, ls="--")
        ax.grid(alpha=0.25)
        if r_ == 0:
            ax.set_title(title)
        if c_ == 0:
            ax.set_ylabel(ylab)
        if r_ == 3:
            ax.set_xlabel("seconds ahead")
from matplotlib.lines import Line2D  # noqa: E402

fig.legend(
    handles=[
        Line2D([], [], color=TEAM["blue"], lw=1.8, label="Blue team: real"),
        Line2D([], [], color=TEAM["blue"], lw=1.8, ls="--", label="Blue team: model"),
        Line2D([], [], color=TEAM["red"], lw=1.8, label="Red team: real"),
        Line2D([], [], color=TEAM["red"], lw=1.8, ls="--", label="Red team: model"),
        Line2D([], [], color="black", lw=1.8, label="Distance: real"),
        Line2D([], [], color="black", lw=1.8, ls="--", label="Distance: model"),
    ],
    loc="upper center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 1.04),
)
fig.tight_layout()
save(fig, "example_predictions")

In [ ]:
# 3 · rolling 4 s forecasts over whole battles
roll = {}
for ep, g in val.groupby(level=0):
    Xg, Sg, Rg, Ug = (g[c].to_numpy() for c in (STATE, SHAPE, REACH_COLS, CTRL))
    n_ = len(g)
    fcs, errs = [], []
    for k0 in range(8, n_ - 4, 8):
        steps_ = min(32, n_ - 1 - k0)
        Yf = current.simulate(Xg[k0][None], Sg[k0][None], Ug[k0 : k0 + steps_][None], reach0=Rg[k0][None])[0]
        fcs.append((k0, Yf[:, :2]))
        if steps_ == 32:
            errs.append(np.abs(Yf[-1, :2] - Xg[k0 + 32, :2]).mean())
    if len(errs) >= 3:
        roll[ep] = (np.mean(errs), fcs, Xg[:, :2], g["t"].to_numpy())
order_b = sorted(roll, key=lambda e: roll[e][0])
print(f"{len(order_b)} battles with at least three full 4 s forecasts")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, (title, q) in zip(axes, RANKS, strict=True):
    ep = order_b[int(q * len(order_b))]
    err, fcs, real, t_ = roll[ep]
    print(
        f"{title.replace('at 4 s ', '')}: battle {ep}, "
        f"average health error after 4 s {err:.3f}, {t_[-1]:.1f} s long"
    )
    for j, team in enumerate(("blue", "red")):
        ax.plot(t_, real[:, j], color=TEAM[team], lw=2.4)
        for k0, f in fcs:
            ax.plot(t_[k0 : k0 + len(f)], f[:, j], color=TEAM[team], lw=1.1, alpha=0.5)
            ax.plot(t_[k0], f[0, j], "o", color=TEAM[team], ms=3)
    ax.set(
        title=title.replace("error at 4 s", "forecast error"),
        xlabel="seconds into the battle",
        xlim=(0, t_[-1]),
        ylim=(0, None),
    )
    ax.grid(alpha=0.25)
axes[0].set_ylabel("health (1 = full team)")
fig.legend(
    handles=[
        Line2D([], [], color=TEAM["blue"], lw=2.4, label="Blue team: real health"),
        Line2D([], [], color=TEAM["blue"], lw=1.1, alpha=0.5, label="Blue team: 4 s forecasts"),
        Line2D([], [], color=TEAM["red"], lw=2.4, label="Red team: real health"),
        Line2D([], [], color=TEAM["red"], lw=1.1, alpha=0.5, label="Red team: 4 s forecasts"),
    ],
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.08),
)
fig.tight_layout()
save(fig, "rolling_forecasts")

## Questions for you

1. Section 2: if an examiner asks "what did SINDy actually discover?", which rows would you point
   to, and which would you call your own design?
2. Section 3: the generic SINDYc model has no physics but similar errors. Where in the plots does it
   behave unphysically, if anywhere?
3. Section 5: which differences between models are larger than their intervals?
4. Section 7: does the model get the *order* of the intents right, even where the levels are off?